In [1]:
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.classification import GBTClassifier, RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator


# Create a SparkSession
spark = SparkSession.builder \
    .appName("CC_Fraud") \
    .config("spark.executor.instances", "2") \
    .getOrCreate()


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/27 12:07:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# Load your dataset into a Spark DataFrame
df = spark.read.csv("../data/clean_train.csv", header=True, inferSchema=True)
df = df.drop("_c0", "trans_num")
# Define your features and target column
feature_columns = [col for col in df.columns if col != "is_fraud"]
label_column = "is_fraud"

In [4]:
df.head(2)

[Row(amt=299.33, lat=36.5422, long=-88.3319, city_pop=1480, unix_time=1355448529, merch_lat=35.980221, merch_long=-88.482625, is_fraud=1, merchant_label=428, category_label=4, gender_label=1, job_label=125),
 Row(amt=7.03, lat=44.0385, long=-123.0614, city_pop=191096, unix_time=1341402508, merch_lat=43.617853, merch_long=-123.611212, is_fraud=0, merchant_label=310, category_label=12, gender_label=0, job_label=414)]

In [5]:
assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")
classifier = RandomForestClassifier(featuresCol="features", labelCol=label_column, numTrees=10, maxBins=700)

pipeline = Pipeline(stages=[assembler, classifier])

In [6]:
# Split data into training and test sets
train_data, test_data = df.randomSplit([0.8, 0.2], seed=42)

In [7]:
df.head()

Row(amt=299.33, lat=36.5422, long=-88.3319, city_pop=1480, unix_time=1355448529, merch_lat=35.980221, merch_long=-88.482625, is_fraud=1, merchant_label=428, category_label=4, gender_label=1, job_label=125)

In [8]:
# Train the pipeline
model = pipeline.fit(train_data)

26/06/27 12:08:18 WARN MemoryStore: Not enough space to cache rdd_44_3 in memory! (computed 19.7 MiB so far)
26/06/27 12:08:18 WARN BlockManager: Persisting block rdd_44_3 to disk instead.
26/06/27 12:08:18 WARN MemoryStore: Not enough space to cache rdd_44_6 in memory! (computed 19.7 MiB so far)
26/06/27 12:08:18 WARN BlockManager: Persisting block rdd_44_6 to disk instead.
26/06/27 12:08:18 WARN MemoryStore: Not enough space to cache rdd_44_8 in memory! (computed 19.7 MiB so far)
26/06/27 12:08:18 WARN BlockManager: Persisting block rdd_44_8 to disk instead.
26/06/27 12:08:18 WARN MemoryStore: Not enough space to cache rdd_44_4 in memory! (computed 19.7 MiB so far)
26/06/27 12:08:18 WARN BlockManager: Persisting block rdd_44_4 to disk instead.
26/06/27 12:08:18 WARN MemoryStore: Not enough space to cache rdd_44_2 in memory! (computed 19.7 MiB so far)
26/06/27 12:08:18 WARN MemoryStore: Not enough space to cache rdd_44_7 in memory! (computed 19.7 MiB so far)
26/06/27 12:08:18 WARN Blo

In [9]:
# Make predictions on test data
predictions = model.transform(test_data)

# Evaluate the model
evaluator = MulticlassClassificationEvaluator(labelCol=label_column, predictionCol='prediction', metricName='accuracy')
accuracy = evaluator.evaluate(predictions)
print("Accuracy:", accuracy)


[Stage 21:>                                                       (0 + 10) / 10]

Accuracy: 0.9957520264184929


In [10]:
model.write().overwrite().save("../models/fraud_pipeline")

In [13]:
spark.stop()